In [77]:
# Install required packages
%pip install gliner transformers datasets huggingface_hub torch accelerate
%pip install sentencepiece protobuf

# Import libraries
import json
import jsonlines
import torch
from gliner import GLiNERConfig, GLiNER
from gliner.training import Trainer, TrainingArguments
from gliner.data_processing.collator import DataCollatorWithPadding, DataCollator
from gliner.utils import load_config_as_namespace
from gliner.data_processing import WordsSplitter, GLiNERDataset
from datasets import Dataset
import numpy as np
from sklearn.model_selection import train_test_split
from huggingface_hub import login, HfApi
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("All packages imported successfully!")

  Using cached transformers-4.56.0-py3-none-any.whl.metadata (40 kB)
  Using cached transformers-4.51.0-py3-none-any.whl.metadata (38 kB)
Using cached transformers-4.51.0-py3-none-any.whl (10.4 MB)

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
All packages imported successfully!


In [78]:
# Load your pre-converted GLiNER data
import json

# Load the converted data
with open('../data/training/gliner_converted_data.json', 'r', encoding='utf-8') as f:
    gliner_data = json.load(f)

print(f"Loaded {len(gliner_data)} examples in GLiNER format")

# Show sample of your data
print("\nSample data structure:")
print(json.dumps(gliner_data[0], indent=2))

# Get entity labels from your data
entity_labels = list(set([entity[2] for item in gliner_data for entity in item['ner']]))
print(f"\nEntity labels: {entity_labels}")

# Check data quality
examples_with_entities = sum(1 for item in gliner_data if len(item['ner']) > 0)
total_entities = sum(len(item['ner']) for item in gliner_data)
print(f"Examples with entities: {examples_with_entities}")
print(f"Total entities found: {total_entities}")
print(f"Average entities per example: {total_entities / len(gliner_data):.2f}")

Loaded 501 examples in GLiNER format

Sample data structure:
{
  "tokenized_text": [
    "What",
    "is",
    "the",
    "total",
    "discount",
    "amount",
    "offered",
    "during",
    "the",
    "clearance",
    "sale",
    "last",
    "quarter",
    "?"
  ],
  "ner": [
    [
      4,
      4,
      "MEASURE"
    ],
    [
      11,
      12,
      "TIMEFRAME"
    ],
    [
      9,
      10,
      "FILTER"
    ]
  ]
}

Entity labels: ['DIMENSION', 'MEASURE', 'FILTER', 'TIMEFRAME']
Examples with entities: 501
Total entities found: 1303
Average entities per example: 2.60


In [79]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "true"

import torch
from gliner import GLiNERConfig, GLiNER
from gliner.training import Trainer, TrainingArguments
from gliner.data_processing.collator import DataCollatorWithPadding, DataCollator
from gliner.utils import load_config_as_namespace
from gliner.data_processing import WordsSplitter, GLiNERDataset

device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

# Create a new GLiNER model
model = GLiNER.from_pretrained("urchade/gliner_small")

# Create data collator
data_collator = DataCollator(model.config, data_processor=model.data_processor, prepare_labels=True)

# Move model to device
model.to(device)
print("Model loaded and moved to device")

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 33893.37it/s]


Model loaded and moved to device


In [80]:
# Convert your data to GLiNER dataset format
from gliner.data_processing import GLiNERDataset
from sklearn.model_selection import train_test_split

# Split into train/validation
train_data, val_data = train_test_split(gliner_data, test_size=0.2, random_state=42)

# Create GLiNER datasets
train_dataset = GLiNERDataset(
    examples=train_data,
    config=model.config,
    tokenizer=model.data_processor.transformer_tokenizer,
    data_processor=model.data_processor
)

val_dataset = GLiNERDataset(
    examples=val_data,
    config=model.config,
    tokenizer=model.data_processor.transformer_tokenizer,
    data_processor=model.data_processor
)

print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(val_data)}")
print("Datasets created successfully!")

100%|██████████| 400/400 [00:00<00:00, 1014955.60it/s]


Total number of entity classes:  4


100%|██████████| 101/101 [00:00<00:00, 788872.82it/s]

Total number of entity classes:  4
Training examples: 400
Validation examples: 101
Datasets created successfully!


In [81]:
# Training arguments
training_args = TrainingArguments(
    output_dir="models",
    learning_rate=5e-6,
    weight_decay=0.01,
    others_lr=1e-5,
    others_weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_ratio=0.1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    focal_loss_alpha=0.75,
    focal_loss_gamma=2,
    num_train_epochs=3,
    eval_strategy="steps",
    save_steps=50,
    save_total_limit=5,
    dataloader_num_workers=0,
    use_cpu=True,  # Set to True for CPU training
    report_to="none",
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=model.data_processor.transformer_tokenizer,
    data_collator=data_collator,
)

print("Training setup complete!")

Training setup complete!


In [82]:
# Start training
print("Starting training...")
trainer.train()
print("Training completed!")

Starting training...


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


KeyError: 'ner'

In [ ]:
# Load your trained model
trained_model = GLiNER.from_pretrained("models/checkpoint-50", load_tokenizer=True)

# Test on a sample question
test_text = "How many customers interacted with the loyalty program in the last month?"

# Your entity labels
labels = ["MEASURE", "DIMENSION", "TIMEFRAME", "FILTER"]

# Perform entity prediction
entities = trained_model.predict_entities(test_text, labels, threshold=0.5)

# Display predicted entities and their labels
for entity in entities:
    print(entity["text"], "=>", entity["label"])

In [ ]:
# Evaluate the model
print("Evaluating model performance...")
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

# Test on a few examples
test_texts = [
    "What is the total revenue from our top 5 products last quarter?",
    "Show me the average customer acquisition cost by channel in Q1 2024",
    "How many customers interacted with the loyalty program last month?"
]

print("\nTesting model predictions:")
for text in test_texts:
    entities = trained_model.predict_entities(text, labels, threshold=0.5)
    print(f"\nText: {text}")
    print(f"Predicted entities: {entities}")

In [ ]:
# Login to Hugging Face
print("Logging into Hugging Face...")
login()  # This will prompt for your token

# Prepare model card
model_card = """
# GLiNER BI Intent Recognition Model

This model is trained to recognize Business Intelligence (BI) entities in natural language questions.

## Entity Types
- **MEASURE**: Metrics like revenue, sales, customers, etc.
- **DIMENSION**: Categories like products, regions, time periods, etc.
- **TIMEFRAME**: Time references like "last quarter", "Q1 2024", etc.
- **FILTER**: Constraints like "top 5", "excluding returns", etc.

## Training Data
- Trained on 100 annotated BI questions
- Uses GLiNER approach for entity recognition
- Base model: microsoft/DialoGPT-medium

## Usage
```python
from transformers import AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained("ssuki/gliner-bi-intent")
model = AutoModelForTokenClassification.from_pretrained("ssuki/gliner-bi-intent")

# Use for entity extraction in BI questions
```
"""

# Save model card
with open("./gliner_bi_intent_model_final/README.md", "w") as f:
    f.write(model_card)

print("Model card created!")# Login to Hugging Face
print("Logging into Hugging Face...")
login()  # This will prompt for your token

# Prepare model card
model_card = """
# GLiNER BI Intent Recognition Model

This model is trained to recognize Business Intelligence (BI) entities in natural language questions.

## Entity Types
- **MEASURE**: Metrics like revenue, sales, customers, etc.
- **DIMENSION**: Categories like products, regions, time periods, etc.
- **TIMEFRAME**: Time references like "last quarter", "Q1 2024", etc.
- **FILTER**: Constraints like "top 5", "excluding returns", etc.

## Training Data
- Trained on 100 annotated BI questions
- Uses GLiNER approach for entity recognition
- Base model: microsoft/DialoGPT-medium

## Usage
```python
from transformers import AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained("ssuki/gliner-bi-intent")
model = AutoModelForTokenClassification.from_pretrained("ssuki/gliner-bi-intent")

# Use for entity extraction in BI questions
```
"""

# Save model card
with open("./gliner_bi_intent_model_final/README.md", "w") as f:
    f.write(model_card)

print("Model card created!")

In [ ]:
# Upload to Hugging Face Hub
from huggingface_hub import HfApi

api = HfApi()

# Create repository
repo_name = "gliner-bi-intent"
username = "ssuki"
full_repo_name = f"{username}/{repo_name}"

print(f"Creating repository: {full_repo_name}")

# Push the model
print("Pushing model to Hugging Face Hub...")
api.upload_folder(
    folder_path="./gliner_bi_intent_model_final",
    repo_id=full_repo_name,
    repo_type="model"
)

print(f"Model successfully uploaded to: https://huggingface.co/{full_repo_name}")
print("You can now use your model with:")
print(f"from transformers import AutoTokenizer, AutoModelForTokenClassification")
print(f"tokenizer = AutoTokenizer.from_pretrained('{full_repo_name}')")
print(f"model = AutoModelForTokenClassification.from_pretrained('{full_repo_name}')")

In [ ]:
# Test the uploaded model
print("Testing the uploaded model...")

# Load from Hugging Face
uploaded_tokenizer = AutoTokenizer.from_pretrained(full_repo_name)
uploaded_model = AutoModelForTokenClassification.from_pretrained(full_repo_name)

# Test prediction
test_text = "What is the total revenue from our top 10 products last year?"
entities = predict_entities(test_text, uploaded_model, uploaded_tokenizer, label2id, id2label)

print(f"\nTest text: {test_text}")
print(f"Predicted entities: {entities}")
print("Model loaded from Hugging Face successfully!")